# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# metadata is a dataclass object (do not treat as dict)
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s as defined by the Croissant schema.

We will iterate over the dataset's record sets, displaying their `@id` (unique identifiers), any fields within those record sets (and their `@id`s), along with a small sample of data for preview.


In [ ]:
# List and inspect RecordSets and their Fields (@id)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("RecordSets found in dataset (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"  RecordSet @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields in this RecordSet:")
        for field in rs.fields:
            print(f"      Field @id: {field.id} | Name: {field.name}")
    else:
        print("    No fields found in this RecordSet.")
    print("")
# Sample a few records from each RecordSet for preview (only first 2 records for brevity)
for rs_id in record_set_ids:
    print(f"First two records from RecordSet '{rs_id}':")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i==1:
            break
    print("")

## 3. Data Extraction
Load data from specific RecordSets into Pandas DataFrames for analysis. Below, we use the RecordSet `@id`s listed above. **All references are by `@id` as required by the Croissant specification.**

Replace with relevant RecordSet IDs found above for exploration.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only create DataFrame if there is data
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame for RecordSet '{rs_id}' has columns: {df.columns.tolist()}")
        print(df.head())
        print("")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using the loaded DataFrames.

We will select one RecordSet and a numeric field (`@id`) in that set for demonstration. **Please adapt the `record_set_id` and `numeric_field_id` variables to match your use-case.**

Here, we use the first available DataFrame and select a numeric field, if present.

In [ ]:
# Choose a RecordSet and a numeric field (by @id)
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to auto-detect a numeric column
    numeric_field_id = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
            # Try converting to numeric (if all values are strings of numbers)
            pd.to_numeric(df[col])
            numeric_field_id = col
            df[col] = pd.to_numeric(df[col], errors='coerce')
            break
        except Exception:
            continue
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0
        print(f"Using numeric field '{numeric_field_id}' and threshold {threshold:0.4f}\n")

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:0.4f} (showing up to 5):")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
            / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records (showing up to 5):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try auto-detect a group field (categorical)
        group_field = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < 20:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field_id].mean()
            print(f"\nGrouped data by '{group_field}' (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.\n")
    else:
        print("No numeric field found for EDA in selected RecordSet.")
else:
    print("No DataFrames available. Please check your record set IDs and data loading.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`. The example plots the normalized numeric field and a histogram of the chosen column.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of '{numeric_field_id}' (filtered, RecordSet: {record_set_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Normalized distribution
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f'{numeric_field_id}_normalized'].dropna(), bins=30, kde=True)
    plt.title(f"Normalized '{numeric_field_id}' Distribution (filtered, RecordSet: {record_set_id})")
    plt.xlabel(f'{numeric_field_id}_normalized')
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No available numeric field with data for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated step-by-step loading, exploration, and processing of a FAIR-compliant dataset using the `mlcroissant` library. All entity references (RecordSets, Fields, Columns) are made using their unique `@id`, ensuring consistency and reproducibility. You can adapt this workflow for your own Croissant schema datasets, customizing the analysis and visualization sections according to your research goals.